# Example

In [1]:
from dotenv import load_dotenv
import os

load_dotenv()
API_KEY = os.getenv('DEEPSEEK_API_KEY')
LIGHTRAG_API = os.getenv('LIGHTRAG_API')

In [2]:
import requests
from langchain.tools import tool
from typing import Literal

# @tool
def _lightrag_retrieve(query: str, query_mode:Literal['local', 'global', 'hybrid', 'naive', 'mix'] = 'mix') -> str:
    """
    Отправляет запрос к API LightRAG и возвращает текстовый ответ.
    LightRAG хранит данные о документах стратегического и территориального планирования, а также градостроительных и нормативных документах.
    
    Parameters
    ----------
    query : str
        Текстовый запрос, который необходимо отправить в LightRAG.
        Может быть любой строкой, с которой модель или система должна работать.
    
    query_mode : Literal['local', 'global', 'hybrid', 'naive', 'mix'], optional
        Режим обработки запроса, который определяет стратегию поиска или генерации ответа:
        - 'local'  : Находит сначала ключевые сущности из знаний и строит контекст вокруг их непосредственных связей в графе.
        - 'global' : Опирается на связи между сущностями по всему графу, подбирая взаимосвязанные отношения (relationships), а затем соответствующие куски.
        - 'hybrid' : Комбинирует local + global подходы, т.е. одновременно локальные сущности и глобальные связи.
        - 'naive'  : Выполняет обычный векторным поиском по фрагментам текста (chunks) без использования графа знаний.
        - 'mix'    : Объединяет данные из графа знаний и обычный векторный поиск по фрагментам (по умолчанию).
    
    Returns
    -------
    str
        Текстовый ответ от API LightRAG.
    
    Example
    -------
    >>> result = lightrag_retrieve("Что такое стратегия социально-экономического развития?", query_mode='global')
    >>> print(result)
    'Стратегия социально-экономического развития...'
    """
    response = requests.post(
        f'{LIGHTRAG_API}/query',
        json={
            'query': query,
            'query_mode': query_mode
        }
    )
    return response.text

lightrag_retrieve = tool(_lightrag_retrieve)

In [3]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="deepseek-chat",
    api_key=API_KEY,
    base_url="https://api.deepseek.com",
    temperature=0.0,
)

## LangGraph

In [4]:
from langchain.agents import create_agent
from pydantic import BaseModel, Field
from abc import ABC
import json

class Model(ABC, BaseModel):

    def __str__(self):
        schema = self.model_json_schema()
        data = self.model_dump()
        return json.dumps({
            'schema': schema,
            'data': data
        })
    
class QueryModel(Model):
    """
    Информация о запросе пользователя по разработке генерального плана населенного пункта
    """
    settlement_name : str = Field(description="Название населенного пункта")
    settlement_type : str = Field(description='Тип населенного пункта')
    current_year : int = Field(description='Текущий год')

### Documents graph

Анализ документов

In [ ]:
from typing import Literal

class Document(Model):
    name : str = Field(description='Название документа')

class StrategicDocument(Document):
    ...

class InheritedDocumentsModel(Model):
    """
    Перечень актуальных документов стратегического и территориального планирования, нормативные и градостроительные документов, которые необходимо учитывать при разработке генерального плана.
    """
    strategic_documents : list[str] = Field(description='Перечень документов стратегического планирования')
    territorial_schemas : list[str] = Field(description='Перечень схем территориального планирования')
    normative_documents : list[str] = Field(description='Перечень региональных и местных нормативов градостроительного проектирования')
    other_documents : list[str] = Field(description='Перечень документов, не относящихся к другим категориям')

class DocumentsState(Model):
    query : QueryModel
    inherited_documents : InheritedDocumentsModel | None = None

In [6]:
def documents_master_node(state : DocumentsState):
    agent = create_agent(
        model=llm,
        tools=[lightrag_retrieve],
        response_format=InheritedDocumentsModel,
    )
    response = agent.invoke({'messages': f"{state.query}"})
    return {
        'inherited_documents': response['structured_response']
    }

In [7]:
from langgraph.graph import StateGraph, START, END

documents_graph = StateGraph(DocumentsState)

documents_graph.add_node('documents_master', documents_master_node)

documents_graph.add_edge(START, 'documents_master')
documents_graph.add_edge('documents_master', END)

documents_app = documents_graph.compile()

In [8]:
documents_result = documents_app.invoke({'query': QueryModel(
    settlement_name='Гатчина',
    settlement_type='Город',
    current_year=2026
)})

In [9]:
documents_result['inherited_documents'].model_dump()

{'strategic_documents': ['Стратегия социально-экономического развития Ленинградской области',
  'Стратегия социально-экономического развития Гатчинского муниципального района',
  'Стратегия социально-экономического развития муниципального образования «Город Гатчина»',
  'Документы стратегического планирования Российской Федерации'],
 'territorial_schemas': ['Схема территориального планирования Ленинградской области (утверждена постановлением Правительства ЛО от 29.12.2012 № 460)',
  'Схема территориального планирования Санкт-Петербургской агломерации',
  'Схема территориального планирования двух и более субъектов РФ (для координации развития Санкт-Петербурга и Ленинградской области)'],
 'normative_documents': ['Градостроительный кодекс Российской Федерации',
  'Региональные нормативы градостроительного проектирования Ленинградской области (РНГП ЛО)',
  'Местные нормативы градостроительного проектирования Ленинградской области (утверждаются Правительством ЛО)',
  'Областной закон Ленинг

### Analysis graph

Анализ экономического и пространственного потенциала территории согласно методическим рекомендациям

XVII. КОНЦЕПЦИЯ ПРОСТРАНСТВЕННОГО РАЗВИТИЯ МО (МАСТЕР-ПЛАН)

In [ ]:
class SocioEconomicModel(Model):
    """
    Социально-экономический анализ
    """
    demography : str = Field(description='Демография')
    productivity : str = Field(description='Производительность труда')
    grdp : str = Field(description='Структура и динамика валового городского продукта')
    labor : str = Field(description='Рынок труда')
    real_estate : str = Field(description='Рынок жилья и коммерческой недвижимости')
    budget : str = Field(description='Бюджетная обеспеченность')

class SpatialModel(Model):
    """
    Пространственный анализ
    """
    functional_organization : str = Field(description='Функциональная организация территории')
    land_tenure : str = Field(description='Структура землевладения')
    ecology : str = Field(description='Природно-экологический каркас')
    architecture : str = Field(description='Архитектурно-градостроительная среда')

class TransportModel(Model):
    """
    Транспортный анализ
    """
    transport : str = Field(description='Городской и внешний пассажирский и грузовой транспорт')
    parking : str = Field(description='Парковки')
    pedestrian : str = Field(description='Пешеходные зоны')

class EngineeringModel(Model):
    """
    Анализ инженерной инфраструктуры
    """
    water : str = Field(description='Водоснабжение и водоотведение')
    energy : str = Field(description='Энергоснабжение')
    heat : str = Field(description='Теплоснабжение')

class AnalysisModel(Model): #XVII. КОНЦЕПЦИЯ ПРОСТРАНСТВЕННОГО РАЗВИТИЯ МО (МАСТЕР-ПЛАН)
    """
    Диагностика экономического и пространственного состояния территории
    """
    socio_economic : SocioEconomicModel | None = Field(default=None, description='Социально-экономический анализ')
    spatial : SpatialModel | None = Field(default=None, description='Пространственный анализ')
    transport : TransportModel | None = Field(default=None, description='Транспортный анализ')
    engineering : EngineeringModel | None = Field(default=None, description='Анализ инженерной инфраструктуры')

class AnalysisState(AnalysisModel):
    query : QueryModel

In [15]:
def create_analysis_node(name : str, model : type[Model]):
    def analysis_node(state : AnalysisState):
        agent = create_agent(
            model=llm,
            tools=[lightrag_retrieve],
            response_format=model,
        )
        response = agent.invoke({'messages': f"{state.query}"})
        return {
            name: response['structured_response']
        }
    return analysis_node

In [ ]:
from langgraph.graph import StateGraph, START, END

analysis_graph = StateGraph(AnalysisState)

analysis_nodes = {name: create_analysis_node(name,model) for name,model in {
    'socio_economic': SocioEconomicModel,
    'spatial': SpatialModel,
    'transport': TransportModel,
    'engineering': EngineeringModel
}.items()}

for name,node in analysis_nodes.items():
    analysis_graph.add_node(name, node)
    analysis_graph.add_edge(START, name)
    analysis_graph.add_edge(name, END)

analysis_app = analysis_graph.compile()

In [ ]:
analysis_result = analysis_app.invoke({'query': QueryModel(
    settlement_name='Гатчина',
    settlement_type='Город'
)})

In [13]:
AnalysisModel(**analysis_result).model_dump()

{'socio_economic': {'demography': 'Демографическая ситуация в Гатчине сложная: численность населения 94,447 человек (на 01.01.2018), естественная убыль населения (-7.5 на 1000), миграционный отток (48% от общей убыли по району в 2017 году). Прогноз: 94.8 тыс. к 2020 году, 95.7 тыс. к 2030 году.',
  'productivity': 'Город является ведущим промышленным центром района - 44% отгрузки предприятий обрабатывающих производств всего Гатчинского муниципального района. Целевой показатель динамики объема отгруженных товаров: рост со 112.6% в 2020 году до 116.8% в 2030 году (среднегодовой индекс).',
  'grdp': 'Прямых данных о Валовом городском продукте Гатчины нет. Косвенные показатели: город обеспечивает значительную долю налоговых доходов Гатчинского муниципального района, является мощным промышленным, научным и административным полюсом роста.',
  'labor': 'Экономически активное население: 56.6 тыс. человек (2017). Занятость: 46,709 человек. Уровень заработной платы: 95.7% от среднего по Ленингра